# Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import datetime
from pyquaternion import Quaternion
import plotly.express as px
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
import argparse

# Utilities

In [ ]:
############### General ###############
def remove_outlier (arr):
  mean = np.mean(arr, axis=0)
  std = np.std(arr, axis=0)
  final_list = [x for x in arr if (x > mean - 2 * std)]
  final_list = [x for x in final_list if (x < mean + 2 * std)]
  return final_list

############### Euler ###############
def euler_diff (arr, time):
  diff = []
  for i in range(len(arr)-1):
    diff.append((np.array(arr[i+1])-np.array(arr[i]))/time)
  return diff

def euler_diff_cir (arr, time):
  diff = []
  for i in range(len(arr)-1):
    diff.append(euler_diff_cir_array(arr[i], arr[i+1])/time)
  return diff
  
def euler_diff_cir_array (arr1, arr2):
  res = np.array([0,0,0])
  if (len(arr1) != len(arr2)):
    print("Error: length of two arrays are not equal")
    return
  for i in range(len(arr1)):
    res[i] = euler_diff_cir_element(arr1[i], arr2[i])
  return res

def euler_diff_cir_element (a, b):
  if (b - a > 180):
    return (b-360)-a
  elif (b - a < -180):
    return b-(a-360)
  else:
    return b-a

def euler_distance (arr, threshold = 0):
  sum = 0
  for i in range(len(arr)-1):
    if abs(arr[i+1]-arr[i]) > threshold:
      sum += abs(arr[i+1]-arr[i])
  return sum

def euler_circle (arr):
  circle = []
  for i in range(len(arr)):
    if (arr[i] > 180):
      circle.append(arr[i]-360)
    else:
      circle.append(arr[i])
  return circle

def euler_negative (arr):
  neg = []
  for i in range(len(arr)):
    neg.append(-arr[i])
  return neg

############### Quaternion ###############
def quaternion_distance (arr, threshold = 0):
  sum = 0
  for i in range(len(arr)-1):
    if abs(Quaternion.absolute_distance(Quaternion(arr[i]), Quaternion(arr[i+1]))) > threshold:
      sum += Quaternion.absolute_distance(Quaternion(arr[i]), Quaternion(arr[i+1]))
  return sum

def quaternion_diff_abs (arr, time):
  diff = []
  for i in range(len(arr)-1):
    diff.append(Quaternion.absolute_distance(Quaternion(arr[i]), Quaternion(arr[i+1]))/time)
  return diff

def quaternion_diff (arr, time):
  diff = []
  for i in range(len(arr)-1):
    diff.append(Quaternion.distance(Quaternion(arr[i]), Quaternion(arr[i+1]))/time)
  return diff

In [ ]:
fov_list = [
  "90", "110", "130", "150", "170"
]

# Files

In [ ]:
userList = [
  'P001',
  'P003',
  'P004',
  'P007',
  'P011',
  'P012',
  'P014',
  'P016',
  'P017',
  'P020',
  'P021',
  'P022',
  'P023',
  'P024',
  'P031'
]

In [ ]:
userIdx = 0

In [ ]:
path = '../data/uist24/pilot/'
filename = 'LOG_Pilot(RE)_' + userList[userIdx] + '.txt'

# Data Input

In [ ]:
username = userList[userIdx]

fov_index = 0
headAngle_qu = [np.array([[0,0,0,0]])] * 5
headAngle_eu = [np.array([[0,0,0]])] * 5
rightHandPosition = [np.array([[0,0,0]])] * 5
leftHandPosition = [np.array([[0,0,0]])] * 5
rightHandAngle_qu = [np.array([[0,0,0,0]])] * 5
leftHandAngle_qu = [np.array([[0,0,0,0]])] * 5
rightHandAngle_eu = [np.array([[0,0,0]])] * 5
leftHandAngle_eu = [np.array([[0,0,0]])] * 5
position = [np.array([[0,0,0]])] * 5

completeFileName = path + filename

f = open(completeFileName, 'r')

count = 0
endedTime = -1

for i, line in enumerate(f):
  if (line.find("startedTime") == 0):
      startedTime = datetime.datetime.strptime(line[line.find(":")+1:].strip('\n'), '%m/%d/%Y %I:%M:%S %p')
  elif (line.find("endedTime") == 0):
      endedTime = datetime.datetime.strptime(line[line.find(":")+1:].strip('\n'), '%m/%d/%Y %I:%M:%S %p')
      break
  elif (line.find("*") == 0):
    if (line.find("*Lens:") == 0):
      # print(line)
      if line.split(": ")[1].strip('\n') == '90':
        fov_index = 0
      elif line.split(": ")[1].strip('\n') == '110':
        fov_index = 1
      elif line.split(": ")[1].strip('\n') == '130':
        fov_index = 2
      elif line.split(": ")[1].strip('\n') == '150':
        fov_index = 3
      elif line.split(": ")[1].strip('\n') == '170':
        fov_index = 4
  elif (line.find("===") == 0):
    continue
  elif i > 2:
    count += 1
    headAngle_qu[fov_index] = np.append(headAngle_qu[fov_index], [[ float(line.split("%")[0][1:].strip(')').split(",")[0]), 
                                                                    float(line.split("%")[0][1:].strip(')').split(",")[1]), 
                                                                    float(line.split("%")[0][1:].strip(')').split(",")[2]),
                                                                    float(line.split("%")[0][1:].strip(')').split(",")[3]) ]], 
                                                                    axis=0)
    
    headAngle_eu[fov_index] = np.append(headAngle_eu[fov_index], [[ float(line.split("%")[1][1:].strip(')').split(",")[0]), 
                                                                    float(line.split("%")[1][1:].strip(')').split(",")[1]), 
                                                                    float(line.split("%")[1][1:].strip(')').split(",")[2]) ]], 
                                                                    axis=0)

    rightHandPosition[fov_index] = np.append(rightHandPosition[fov_index], [[ float(line.split("%")[2][1:].strip(')').split(",")[0]), 
                                                                              float(line.split("%")[2][1:].strip(')').split(",")[1]), 
                                                                              float(line.split("%")[2][1:].strip(')').split(",")[2]) ]], 
                                                                              axis=0)
    
    leftHandPosition[fov_index] = np.append(leftHandPosition[fov_index], [[ float(line.split("%")[3][1:].strip(')').split(",")[0]), 
                                                                            float(line.split("%")[3][1:].strip(')').split(",")[1]), 
                                                                            float(line.split("%")[3][1:].strip(')').split(",")[2]) ]], 
                                                                            axis=0)
    
    rightHandAngle_qu[fov_index] = np.append(rightHandAngle_qu[fov_index], [[ float(line.split("%")[4][1:].strip(')').split(",")[0]), 
                                                                              float(line.split("%")[4][1:].strip(')').split(",")[1]), 
                                                                              float(line.split("%")[4][1:].strip(')').split(",")[2]),
                                                                              float(line.split("%")[4][1:].strip(')').split(",")[3]) ]], 
                                                                              axis=0)
          
    leftHandAngle_qu[fov_index] = np.append(leftHandAngle_qu[fov_index], [[ float(line.split("%")[5][1:].strip(')').split(",")[0]), 
                                                                            float(line.split("%")[5][1:].strip(')').split(",")[1]), 
                                                                            float(line.split("%")[5][1:].strip(')').split(",")[2]),
                                                                            float(line.split("%")[5][1:].strip(')').split(",")[3]) ]], 
                                                                            axis=0)
    
    rightHandAngle_eu[fov_index] = np.append(rightHandAngle_eu[fov_index], [[ float(line.split("%")[6][1:].strip(')').split(",")[0]), 
                                                                              float(line.split("%")[6][1:].strip(')').split(",")[1]), 
                                                                              float(line.split("%")[6][1:].strip(')').split(",")[2]) ]], 
                                                                              axis=0)
          
    leftHandAngle_eu[fov_index] = np.append(leftHandAngle_eu[fov_index], [[ float(line.split("%")[7][1:].strip(')').split(",")[0]), 
                                                                            float(line.split("%")[7][1:].strip(')').split(",")[1]), 
                                                                            float(line.split("%")[7][1:].strip(')').split(",")[2]) ]], 
                                                                            axis=0)
    
    position[fov_index] = np.append(position[fov_index], [[ float(line.split("%")[8][1:].strip(')\n').split(",")[0]), 
                                                            float(line.split("%")[8][1:].strip(')\n').split(",")[1]), 
                                                            float(line.split("%")[8][1:].strip(')\n').split(",")[2]) ]], 
                                                            axis=0)

In [ ]:
for i in range(5):
  if len(headAngle_qu) > 1:
    headAngle_qu[i] = headAngle_qu[i][1:]
    headAngle_eu[i] = headAngle_eu[i][1:]
    rightHandPosition[i] = rightHandPosition[i][1:]
    leftHandPosition[i] = leftHandPosition[i][1:]
    rightHandAngle_qu[i] = rightHandAngle_qu[i][1:]
    leftHandAngle_qu[i] = leftHandAngle_qu[i][1:]
    rightHandAngle_eu[i] = rightHandAngle_eu[i][1:]
    leftHandAngle_eu[i] = leftHandAngle_eu[i][1:]
    position[i] = position[i][1:]

In [ ]:
if endedTime != -1:
  frame_length = (endedTime - startedTime).total_seconds() / count
else:
  frame_length = 0.07736974

print("frame_length: ", frame_length)

In [ ]:
results = []
for j in range(5):
  result = {
    'user': username,
    'fov': 90 + j * 20,
    'widePortion': -1,
    'transitionPortion': -1,
    'transitionCount': -1,
    'head_qu_distance': -1,
    'head_qu_mean_velocity': -1,
    'head_qu_mean_acc': -1,
    'rightHand_qu_distance': -1,
    'rightHand_qu_mean_velocity': -1,
    'rightHand_qu_mean_acc': -1,
    'leftHand_qu_distance': -1,
    'leftHand_qu_mean_velocity': -1,
    'leftHand_qu_mean_acc': -1,
    'player_mean_velocity': -1,
    'player_mean_acc': -1,
    'hands_mean_distance': -1,
  }
  results.append(result)

# Head

In [ ]:
# euler haed angle
for i in range(5):
  print("in fov", fov_list[i])
  headAngle_distance_eu = [euler_distance(headAngle_eu[i][:, 0]), euler_distance(headAngle_eu[i][:, 1]), euler_distance(headAngle_eu[i][:, 2])]

  th = 1
  headAngle_distance_th_eu = [euler_distance(headAngle_eu[i][:, 0], th), euler_distance(headAngle_eu[i][:, 1], th), euler_distance(headAngle_eu[i][:, 2], th)]

  print("Euler Head Angular Distance")
  print("pitch distance:", round(headAngle_distance_eu[0], 2), "degrees", " |  thresholded:", round(headAngle_distance_th_eu[0], 2), "degrees")
  print("yaw distance:", round(headAngle_distance_eu[1], 2), "degrees", " |  thresholded:", round(headAngle_distance_th_eu[1], 2), "degrees")
  print("roll distance:", round(headAngle_distance_eu[2], 2), "degrees", " |  thresholded:", round(headAngle_distance_th_eu[2], 2), "degrees")

In [ ]:
N = 180

direction = ['Pitch', 'Yaw', 'Roll']

for j in range(5):
  print("in fov", fov_list[j])
  # for i in range(len(direction)):
  i = 1
  theta = np.linspace(0.0, 2 * np.pi, N, endpoint=False)
  radii = np.histogram(headAngle_eu[j][:,i], bins=N, range=[0, 360])[0]
  width = (2*np.pi) / N

  ax = plt.subplot(polar=True)
  bars = ax.bar(theta, radii, width=width, bottom=max(np.histogram(headAngle_eu[j][:,i], bins=N, range=[0, 360])[0])/3, color='slateblue')

  ax.set_title('Euler Head ' + direction[i] + ' Direction Distribution')
  ax.set_xlabel('degree [every ' + str(360/N) + r'$\degree$' + ']')
  ax.set_ylabel('frequency', rotation=0, labelpad=-50)

  plt.show()

  # 0 to 90 degrees means heads down

In [ ]:
for j in range(5):
  print("in fov", fov_list[j])

  distance_quaternion = quaternion_distance(headAngle_qu[j])

  distance_quaternion_threshold = quaternion_distance(headAngle_qu[j], 0)

  print("Quaternion Head Angular Distance:", round(distance_quaternion, 2), "rad", " |  thresholded:", round(distance_quaternion_threshold, 2), "rad")

  results[j].update({'head_qu_distance': distance_quaternion})

In [ ]:
headAngle_velocity_qu = [0] * 5
headAngle_acc_qu = [0] * 5
for j in range(5):
  print("in fov", fov_list[j])
  headAngle_velocity_qu[j] = quaternion_diff(headAngle_qu[j], frame_length)
  print(headAngle_velocity_qu[j]) # scalar value
  headAngle_acc_qu[j] = euler_diff(headAngle_velocity_qu[j], frame_length)
  print(headAngle_acc_qu[j])

In [ ]:
for j in range(5):
  print("in fov", fov_list[j])

  size = (5,3)

  fig, ax = plt.subplots(figsize=size)
  fig.subplots_adjust(bottom=0.2, left=0.2)
  filtered = [x for x in headAngle_velocity_qu[j] if 0 <= x <= 1]
  ax.hist(remove_outlier(headAngle_velocity_qu[j]), bins=100, label='x', color='slateblue')
  ax.set_title('Quaternion Head Direction Velocity Distribution')
  ax.set_xticks(np.arange(0, 1, 0.1))  # Set the x ticks from -10 to 10 with a step of 2
  ax.set_xlim(0, 1)  # Fix the x range from -10 to 10
  ax.set_xlabel('velocity (rad/s)')
  ax.set_ylabel('frequency')
  plt.show()

  print("Average Quaternion Head Direction Velocity", np.mean(headAngle_velocity_qu[j]), "rad/s")
  results[j].update({'head_qu_mean_velocity': np.mean(headAngle_velocity_qu[j])})

In [ ]:
for j in range(5):
  print("in fov", fov_list[j])

  size = (5,3)

  fig, ax = plt.subplots(figsize=size)
  fig.subplots_adjust(bottom=0.2, left=0.2)

  filtered = [x for x in headAngle_acc_qu[j] if -10 <= x <= 10]
  ax.hist(remove_outlier(filtered), bins=100, label='x', color='slateblue')
  
  ax.set_title('Quaternion Head Direction Acceleration Distribution')

  ax.set_xticks(np.arange(-10, 11, 2))  # Set the x ticks from -10 to 10 with a step of 2
  ax.set_xlim(-10, 10)  # Fix the x range from -10 to 10
  ax.set_xlabel('acceleration (rad/s' + r'$^2$' + ')')

  ax.set_ylabel('frequency')
  plt.show()

  print("Average Quaternion Head Direction Acceleration", np.mean(headAngle_acc_qu[j]), "rad/s^2")
  results[j].update({'head_qu_mean_acc': np.mean(headAngle_acc_qu[j])})

# Hands

In [ ]:
distance = [0] * 5
for j in range(5):
  print("in fov", fov_list[j])
  tmp_distance = []
  for i in range(len(leftHandPosition[j])):
    # Calculating the Euclidean distance and appending to tmp_distance
    dist = np.linalg.norm(leftHandPosition[j][i] - rightHandPosition[j][i])
    tmp_distance.append(dist)
  distance[j] = tmp_distance
  print("Hands distance list:", distance[j])

In [ ]:
for j in range(5):
  print("in fov", fov_list[j])
  print("Hands distance mean:", np.mean(distance[j]), "m")
  results[j].update({'hands_mean_distance': np.mean(distance[j])})

  size = (5,3)

  fig, ax = plt.subplots(figsize=size)
  fig.subplots_adjust(bottom=0.2, left=0.2)
  ax.hist(remove_outlier(distance[j]), bins=100, label='x', color='slateblue')
  ax.set_title('Hands Distance Distribution')
  ax.set_xlabel('distance (m)')
  ax.set_ylabel('frequency')
  plt.show()

# Results

In [ ]:
results

In [ ]:
# turn into dataframe
df = pd.DataFrame(results)
df.head()